In [1]:
import pandas as pd
import os
from glob import glob
import numpy as np
from pandas.api.types import CategoricalDtype

In [2]:
df_1 = pd.read_parquet('회원_전처리_Segment.parquet')  
df_2 = pd.read_parquet('신용_전처리_Segment.parquet') 
df_3 = pd.read_parquet('승인_전처리_Segment.parquet')
df_4 = pd.read_parquet('청구_전처리_Segment.parquet')
df_5 = pd.read_parquet('잔액_전처리_Segment.parquet')
df_6 = pd.read_parquet('채널_전처리_Segment.parquet')
df_7 = pd.read_parquet('마케팅_전처리_Segment.parquet')
df_8 = pd.read_parquet('성과_전처리_Segment.parquet')

In [4]:
df_merged = (
    df_1
    .merge(df_2,  on=['ID','Segment'], how='left')
    .merge(df_3,  on=['ID','Segment'], how='left')
    .merge(df_4,  on=['ID','Segment'], how='left')
    .merge(df_5,  on=['ID','Segment'], how='left')
    .merge(df_6,  on=['ID','Segment'], how='left')
    .merge(df_7,  on=['ID','Segment'], how='left')
    .merge(df_8,  on=['ID','Segment'], how='left'))
print(df_merged.shape)
df_merged

(400000, 98)


,ID,Segment,이용거절여부_카드론,탈회횟수_누적,_1순위카드이용건수,이용가능카드수_신용체크,입회일자_신용,이용가능여부_해외겸용_본인,회원여부_이용가능_카드론,Life_Stage,...,변동률_잔액_일시불_B1M,변동률_잔액_B1M,증감율_이용금액_신판_전월,변동률_RV일시불평잔,증감율_이용건수_할부_분기,변동률_일시불평잔,증감율_이용금액_할부_전월,증감율_이용건수_일시불_전월,증감율_이용건수_신용_분기,잔액_신판ca최대한도소진율_r6m
0,TRAIN_000000,D,0.0,1.0,25.46875,2.0,20130101.0,0.0,0.0,5.0,...,0.064887,-0.048348,0.242819,0.999998,0.000000,0.835339,0.000000,-0.031510,-0.292315,0.860117
1,TRAIN_000001,E,0.0,1.0,33.68750,1.0,20170801.0,0.0,1.0,4.0,...,-0.061162,-0.076490,0.168148,0.977866,0.000000,0.929355,0.000000,-0.137449,-0.207193,0.708728
2,TRAIN_000002,C,0.0,1.0,48.40625,2.0,20080401.0,1.0,0.0,6.0,...,0.101498,0.038662,0.297246,0.988566,-0.749999,1.043529,0.537878,0.002922,-0.123001,0.942631
3,TRAIN_000003,D,0.0,1.0,18.12500,3.0,20160501.0,1.0,0.0,5.0,...,0.120753,0.090990,0.300878,0.999998,-0.080086,0.957896,-0.201461,0.050476,0.213005,1.072308
4,TRAIN_000004,E,0.0,1.0,-1.93750,2.0,20180601.0,1.0,1.0,4.0,...,0.000000,0.000000,0.999915,0.999998,0.000000,0.749999,0.000000,1.124998,0.999998,0.002769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,E,0.0,0.0,1.09375,2.0,20010701.0,1.0,1.0,1.0,...,0.000000,0.000000,0.000039,0.999998,0.000000,0.999998,0.000000,0.000000,0.000000,0.018348
399996,TRAIN_399996,D,0.0,1.0,47.40625,1.0,20170701.0,1.0,1.0,5.0,...,-0.115933,-0.144650,0.170532,0.999998,0.000000,0.709682,0.000000,-0.012479,-0.494337,0.194268
399997,TRAIN_399997,C,0.0,0.0,33.25000,1.0,20090501.0,1.0,0.0,6.0,...,0.077845,0.040367,0.264317,0.999998,1.749996,1.048831,-1.062498,-0.011907,-0.067948,0.213335
399998,TRAIN_399998,E,0.0,0.0,-2.00000,1.0,20130101.0,0.0,1.0,4.0,...,0.000000,0.000000,0.000016,0.999998,0.000000,0.999998,0.000000,0.000000,0.000000,0.006487


In [7]:
ex1 = df_merged

In [8]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)


삭제 대상 컬럼 (결측>20% 또는 동일값>80%): []


In [9]:
import pandas as pd
import numpy as np

num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.7]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)


high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 22


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,이용거절여부_카드론,카드론동의여부,1.000000,0.036041,0.036041
1,정상청구원금_B5M,청구금액_R6M,0.920529,0.658360,0.609427
2,이용금액_할부_R12M,잔액_할부_B0M,0.880914,0.351085,0.272251
3,회원여부_이용가능_카드론,월상환론한도금액,0.869894,0.013430,0.049192
4,최종이용일자_일시불,변동률_RV일시불평잔,0.797374,0.094616,0.105983
5,이용금액_R3M_신용체크,정상청구원금_B5M,0.779786,0.632131,0.658360
6,청구서발송여부_B0,상환개월수_결제일_R6M,0.778978,0.215284,0.141026
7,이용가능카드수_신용체크,이용카드수_신용체크,0.776259,0.358194,0.405206
8,CA이자율_할인전,CL이자율_할인전,0.764569,0.125456,0.036413
9,최대이용금액_할부_무이자_R12M,이용금액_할부_R12M,0.747528,0.339048,0.351085


In [10]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))


print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 21
제거할 피처 목록:
['RV최소결제비율', '최종탈회후경과월', 'RP후경과월_교통', '이용가능여부_해외겸용_본인', '_1순위교통업종_이용금액', '포인트_이용포인트_R3M', '청구금액_R6M', '변동률_일시불평잔', '카드론동의여부', '_2순위교통업종_이용금액', '월상환론한도금액', '최종이용일자_일시불', '_2순위카드이용금액', '회원여부_이용가능_카드론', '최대이용금액_할부_무이자_R12M', '상환개월수_결제일_R6M', '잔액_할부_B0M', '이용가능카드수_신용체크', '청구서발송여부_B0', '이용금액_R3M_신용체크', 'CL이자율_할인전']


In [11]:
cols_to_drop = ['RV최소결제비율', '최종탈회후경과월', 'RP후경과월_교통', '이용가능여부_해외겸용_본인', '_1순위교통업종_이용금액', '포인트_이용포인트_R3M', '청구금액_R6M', '변동률_일시불평잔', '카드론동의여부', '_2순위교통업종_이용금액', '월상환론한도금액', '최종이용일자_일시불', '_2순위카드이용금액', '회원여부_이용가능_카드론', '최대이용금액_할부_무이자_R12M', '상환개월수_결제일_R6M', '잔액_할부_B0M', '이용가능카드수_신용체크', '청구서발송여부_B0', '이용금액_R3M_신용체크', 'CL이자율_할인전']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)

In [15]:
ex1.to_parquet('Segment_merge_ver_04.parquet', index=False)

In [16]:
ex1.columns.tolist()

['ID',
 'Segment',
 '이용거절여부_카드론',
 '탈회횟수_누적',
 '_1순위카드이용건수',
 '입회일자_신용',
 'Life_Stage',
 '이용카드수_신용체크',
 '수신거부여부_SMS',
 '이용여부_3M_해외겸용_본인',
 '최종카드발급경과월',
 '유효카드수_체크',
 '수신거부여부_TM',
 '남녀구분코드',
 '일시상환론한도금액',
 'CA이자율_할인전',
 '일시불ONLY전환가능여부',
 '카드이용한도금액',
 '상향가능CA한도금액',
 '쇼핑_기타_이용금액',
 '정상청구원금_B5M',
 '이용후경과월_일시불',
 '증감_RP건수_전월',
 '최종이용일자_CA',
 '교통_택시이용금액',
 '_2순위업종',
 '연속유실적개월수_기본_24M_카드',
 '_1순위교통업종',
 '_1순위납부업종',
 '최대이용금액_할부_유이자_R12M',
 '이용금액_부분무이자_R3M',
 '최종이용일자_할부',
 'RP금액_B0M',
 '연체입금원금_B0M',
 '이용금액_할부_R12M',
 '_1순위업종',
 'RP건수_B0M',
 '_1순위납부업종_이용금액',
 '이용금액_체크_R12M',
 '교통_정비이용금액',
 '_1순위여유업종_이용금액',
 '_2순위교통업종',
 '최종이용일자_체크',
 '_3순위쇼핑업종',
 '_2순위쇼핑업종',
 '_3순위업종',
 '납부_기타이용금액',
 '혜택수혜금액_R3M',
 '청구서수령방법',
 '대표청구지고객주소구분코드',
 '대표결제일',
 '할인금액_청구서_B0M',
 '포인트_적립포인트_R12M',
 '최종연체회차',
 '월중평잔',
 '평잔_일시불_6M',
 '방문횟수_앱_B0M',
 '인입일수_ARS_R6M',
 '불만제기후경과월_R12M',
 '인입후경과월_ARS',
 '컨택건수_이용유도_TM_R6M',
 '컨택건수_보험_TM_R6M',
 '컨택건수_이용유도_인터넷_R6M',
 '컨택건수_이용유도_EM_R6M',
 '컨택건수_이용유도_청구서_B0M',
 '컨택건수_이용유도_LMS_B0M',
 

In [16]:
ex1

,기준년월,ID,Segment,입회일자_신용,수신거부여부_TM,이용카드수_신용체크,최종유효년월_신용_이용가능,이용여부_3M_해외겸용_본인,카드이용한도금액,CA이자율_할인전,...,월중평잔,평잔_일시불_6M,인입일수_ARS_R6M,방문후경과월_앱_R6M,불만제기후경과월_R12M,컨택건수_이용유도_TM_R6M,컨택건수_이용유도_EM_R6M,잔액_신판ca최대한도소진율_r6m,변동률_RV일시불평잔,Segment_encoded
0,201807,TRAIN_000000,D,20130101,0,1,202110.0,0,19354,22.995207,...,17237,2440,8,6,12,3,57,0.849842,0.999998,3
1,201807,TRAIN_000001,E,20170801,0,1,202112.0,0,9996,14.793821,...,7967,2677,0,6,12,2,2,0.851009,1.092698,4
2,201807,TRAIN_000002,C,20080401,0,1,202111.0,0,88193,22.014276,...,59917,9118,1,0,12,2,12,0.938161,1.006124,2
3,201807,TRAIN_000003,D,20160501,0,1,202201.0,1,19062,22.998014,...,27854,884,10,6,12,2,35,1.135424,0.999998,3
4,201807,TRAIN_000004,E,20180601,0,1,202201.0,1,177222,14.661948,...,0,21,0,6,0,7,0,0.000000,0.999998,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,TRAIN_399995,E,20010701,0,1,202110.0,1,20070,15.243670,...,0,0,0,6,0,0,0,0.032439,0.999998,4
2399996,201812,TRAIN_399996,D,20170701,0,1,202110.0,1,84217,14.843464,...,29429,12524,0,6,12,0,58,0.168081,0.999998,3
2399997,201812,TRAIN_399997,C,20090501,1,1,202110.0,1,52612,17.038599,...,7383,3241,0,6,12,0,0,0.190393,0.999998,2
2399998,201812,TRAIN_399998,E,20130101,1,0,202202.0,0,10002,15.182880,...,0,0,0,6,0,0,0,0.012677,0.999998,4
